In [ ]:
# Inspect public API surface.
import pygaljs

public_names = [n for n in dir(pygaljs) if not n.startswith('_')]
print(f"Public names: {public_names}")

# pygaljs exposes a `uri()` helper; verify it's callable.
assert hasattr(pygaljs, 'uri'), "pygaljs.uri not found"
assert callable(pygaljs.uri), "pygaljs.uri is not callable"
print(f"pygaljs.uri = {pygaljs.uri}")

In [ ]:
# Locate the bundled JavaScript assets on disk.
import os
import pygaljs

pkg_dir = os.path.dirname(pygaljs.__file__)
print(f"Package dir: {pkg_dir}")

all_files = []
for root, _dirs, files in os.walk(pkg_dir):
    for f in files:
        all_files.append(os.path.relpath(os.path.join(root, f), pkg_dir))

js_files = [f for f in all_files if f.endswith('.js')]
print(f"Total files: {len(all_files)}")
print(f"JS files bundled: {len(js_files)}")
for j in js_files[:10]:
    print(f"  {j}")

assert len(js_files) > 0, "Expected bundled .js files inside pygaljs"

In [ ]:
# Exercise pygaljs.uri() — it should return a path-like string to a bundled JS file.
import pygaljs

uri = pygaljs.uri('2.0.x', 'pygal-tooltips.js')
print(f"Tooltip JS URI: {uri}")

assert isinstance(uri, str), f"Expected str, got {type(uri)}"
assert uri.endswith('pygal-tooltips.js'), f"Unexpected suffix: {uri}"
print(f"Type: {type(uri).__name__}, length: {len(uri)}")

In [ ]:
# Verify the resolved URI points to an actual file on disk.
import os
import pygaljs

uri = pygaljs.uri('2.0.x', 'pygal-tooltips.js')
# Strip file:// prefix if present
path = uri.replace('file://', '') if uri.startswith('file://') else uri

print(f"Resolved path: {path}")
exists = os.path.exists(path)
print(f"Exists on disk: {exists}")

if exists:
    size = os.path.getsize(path)
    print(f"File size: {size} bytes")
    assert size > 0, "JS file is empty"
else:
    # If exact path doesn't exist, at least confirm SOME version dir does
    pkg_dir = os.path.dirname(pygaljs.__file__)
    js_dir = os.path.join(pkg_dir, 'static')
    print(f"Fallback check — static dir exists: {os.path.isdir(js_dir)}")

In [ ]:
# Sanity: pygaljs is purely a static-asset shim — confirm it has no surprise runtime deps.
import pygaljs
import sys

# After import, the only newly-loaded modules under pygaljs.* should be pygaljs itself.
pygaljs_modules = [m for m in sys.modules if m == 'pygaljs' or m.startswith('pygaljs.')]
print(f"Loaded pygaljs modules: {pygaljs_modules}")

# Final smoke: uri() with different args should yield different strings.
u1 = pygaljs.uri('2.0.x', 'pygal-tooltips.js')
u2 = pygaljs.uri('2.0.x', 'pygal-tooltips.min.js')
print(f"u1 = {u1}")
print(f"u2 = {u2}")
assert u1 != u2, "pygaljs.uri returned identical paths for different args"
print("\npygaljs smoke test passed")